# Gaussian Naive Bayes for numerical inputs

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/people.csv")
df

## An exact value is never repeated

In [ ]:
print("rows with height exactly 185:", (df["height_cm"] == 185).sum())

## Fit one normal distribution per class and column

In [ ]:
def normal_pdf(x, mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))

query = {"height_cm": 185, "weight_lb": 170}
scores = {}
for g, sub in df.groupby("gender"):
    s = len(sub) / len(df)                                   # prior
    for col, x in query.items():
        mu, sd = sub[col].mean(), sub[col].std()             # sample sd (divides by n - 1)
        d = normal_pdf(x, mu, sd)
        print(f"{g:6s} {col}: mean {mu:.2f}, sd {sd:.2f}, density at {x}: {d:.5f}")
        s *= d
    scores[g] = s
total = sum(scores.values())
print({g: f"{s:.3e}" for g, s in scores.items()}, {g: round(s / total, 4) for g, s in scores.items()})

## scikit-learn: GaussianNB

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB().fit(df[["height_cm", "weight_lb"]], df["gender"])
print("class means\n", nb.theta_.round(2), "\nclass variances (divide by n)\n", nb.var_.round(2))
print(dict(zip(nb.classes_, nb.predict_proba(pd.DataFrame([query]))[0].round(4))))